# Методы оптимизации в машинном обучении

### О задании

Настоящее практическое задание посвящено методу градиентного спуска и методу Ньютона. Вам предстоит вывести формулы, реализовать методы оптимизации и провести серии экспериментов.

Задание устроено так: сначала идёт короткий блок теории, а сразу за ним — конкретное задание, для выполнения которого этой теории достаточно. Двигайтесь по порядку: прочитали теорию — выполнили задание — проверили себя тестами — перешли к следующему шагу. Так большое задание распадается на небольшие самостоятельные части.

**Файлы, приложенные к заданию**:

* `optimization.py` — прототипы методов оптимизации: линейный поиск, градиентный спуск, метод Ньютона;
* `oracles.py` — прототипы оракулов (квадратичная функция, логистическая регрессия) и разностных производных;
* `plot_trajectory_2d.py` — утилиты для рисования линий уровня функции и траекторий методов;
* `presubmit_tests.py` — предварительные тесты для самопроверки;
* `data/` — наборы данных *w8a*, *gisette*, *real-sim* для финального эксперимента.

Часть функций уже реализована; те, что нужно реализовать вам, помечены комментариями `# TODO`.

**Как проверять себя по ходу работы.** Тесты запускаются через `pytest` (если он не установлен — `pip install pytest`). После каждого задания запускайте предварительные тесты:

```
python -m pytest presubmit_tests.py
```

Что именно сдавать и как оформлять отчёт — описано в последнем разделе «Оформление и проверка задания».

# Часть 1. Методы спуска и линейный поиск

## 1.1 Общая схема метода спуска

Рассматриваем задачу гладкой безусловной оптимизации:
$$\underset{x \in \mathbb{R}^n}{\min} \; f(x).$$

Методы спуска итеративно строят последовательность точек $(x_k)_{k=0}^{\infty}$ из $\mathbb{R}^n$ по правилу
$$x_{k+1} = x_k + \alpha_k d_k.$$

Число $k = 0, 1, \ldots$ называется номером итерации метода. Скаляр $\alpha_k \geq 0$ называется длиной шага, а вектор $d_k \in \mathbb{R}^n$ — направлением поиска. В методах спуска требуется, чтобы направление поиска $d_k$ являлось направлением спуска для функции $f$ в точке $x_k$, т. е. удовлетворяло неравенству
$$\langle \nabla f(x_k), d_k \rangle < 0.$$

В этом случае можно гарантировать, что для всех достаточно маленьких $\alpha_k$ значение функции $f$ в новой точке $x_{k+1}$ уменьшится:
$$f(x_{k+1}) < f(x_k).$$

Общая схема метода спуска приведена ниже.

**Алгоритм 1.** Общая схема метода спуска  
**Вход:** начальная точка $x_0$; максимальное число итераций $K$.  
1: **for** $k \leftarrow 0$ **to** $K$ **do**  
2: &nbsp;&nbsp;&nbsp;&nbsp;*(Вызов оракула)* Вычислить $f(x_k)$, $\nabla f(x_k)$ и пр.  
3: &nbsp;&nbsp;&nbsp;&nbsp;*(Критерий остановки)* Если выполнен критерий остановки, то выход.  
4: &nbsp;&nbsp;&nbsp;&nbsp;*(Вычисление направления)* Вычислить направление спуска $d_k$.  
5: &nbsp;&nbsp;&nbsp;&nbsp;*(Линейный поиск)* Найти подходящую длину шага $\alpha_k$.  
6: &nbsp;&nbsp;&nbsp;&nbsp;*(Обновление)* $x_{k+1} \leftarrow x_k + \alpha_k d_k$.  
7: **end for**  
**Выход:** последняя вычисленная точка $x_k$.

## 1.2 Критерий остановки

Идеальным критерием остановки в методе является проверка условия $f(x_k) - f^* < \tilde{\varepsilon}$, где $f^*$ — минимальное значение функции $f$. Такой критерий целесообразно использовать, если оптимальное значение функции известно заранее. К сожалению, зачастую это не так, и поэтому нужно использовать другой критерий.

Наиболее популярным является критерий, основанный на норме градиента:
$$\|\nabla f(x_k)\|_2^2 < \tilde{\varepsilon}.$$

Квадрат здесь ставят за тем, что для «хороших» функций невязка по функции $f(x_k) - f^*$ имеет тот же порядок, что и $\|\nabla f(x_k)\|_2^2$, а не $\|\nabla f(x_k)\|_2$ (например, это верно для сильно выпуклых функций с липшицевым градиентом); так, если $\|\nabla f(x_k)\|_2 \sim 10^{-5}$, то $f(x_k) - f^* \sim 10^{-10}$.

Наконец, для того, чтобы критерий не зависел от того, в каких единицах измеряется функция $f$ (т. е. не изменялся при переходе от функции $f$ к функции $tf$, где $t > 0$), имеет смысл использовать следующий относительный вариант критерия:
$$\|\nabla f(x_k)\|_2^2 \leq \varepsilon \|\nabla f(x_0)\|_2^2, \tag{1.1}$$
где $\varepsilon \in (0, 1)$ — заданная относительная точность. Критерий остановки (1.1) гарантирует, что метод уменьшит начальную невязку $\|\nabla f(x_0)\|_2$ в $\varepsilon^{-1}$ раз.

**Во всех методах этого задания нужно использовать именно критерий остановки (1.1).**

## 1.3 Линейный поиск

Рассматривается функция
$$\varphi_k(\alpha) := f(x_k + \alpha d_k).$$
Заметим, что
$$\varphi'_k(\alpha) = \langle \nabla f(x_k + \alpha d_k), d_k \rangle.$$
Поскольку $d_k$ является направлением спуска, то $\varphi'_k(0) = \langle \nabla f(x_k), d_k \rangle < 0$.

**Условием Армихо** для $\alpha$ называется выполнение следующего неравенства:
$$\varphi_k(\alpha) \leq \varphi_k(0) + c_1 \alpha \varphi'_k(0),$$
где $c_1 \in (0, 0.5)$ — некоторая константа.

Для поиска точки $\alpha$, удовлетворяющей условию Армихо, обычно используют метод дробления шага (бэктрекинг):

**Алгоритм 2.** Метод дробления шага  
**Вход:** функция $\varphi_k: \mathbb{R}_+ \to \mathbb{R}$; начальная точка $\alpha_k^{(0)}$.  
1: $\alpha \leftarrow \alpha_k^{(0)}$.  
2: **while** $\varphi_k(\alpha) > \varphi_k(0) + c_1 \alpha \varphi'_k(0)$ **do**  
3: &nbsp;&nbsp;&nbsp;&nbsp;$\alpha \leftarrow \alpha / 2$.  
4: **end while**  
**Выход:** $\alpha$.

«Адаптивный» метод подбора шага запоминает величину $\alpha_k$, найденную на текущей итерации, и на следующей итерации начинает процедуру дробления с $\alpha_{k+1}^{(0)} := 2\alpha_k$. Исключение здесь составляют ньютоновские и квазиньютоновские методы — в этих методах процедуру дробления шага всегда нужно начинать с $\alpha_k^{(0)} := 1$.

**Сильные условия Вульфа:**
$$\varphi_k(\alpha) \leq \varphi_k(0) + c_1 \alpha \varphi'_k(0),$$
$$|\varphi'_k(\alpha)| \leq c_2 |\varphi'_k(0)|,$$
где $c_1 \in (0, 0.5)$, $c_2 \in (c_1, 1)$.

Самостоятельно реализовывать схему для сильных условий Вульфа не нужно — используйте библиотечную функцию `line_search` из модуля `scipy.optimize`: она реализует поиск шага по сильным условиям Вульфа. В ней начальная длина шага $\alpha_k^{(0)}$ автоматически выбирается равной 1.

### Задание 1. Линейный поиск — файл `optimization.py`

**Что нужно сделать.** Реализуйте метод `line_search` класса `LineSearchTool` — процедуру подбора длины шага $\alpha_k$ для заданной точки $x_k$ и направления спуска $d_k$. Метод должен поддерживать три стратегии (поле `method`):

* `'Constant'` — постоянный шаг: просто вернуть фиксированное значение `c`;
* `'Armijo'` — метод дробления шага (Алгоритм 2). Стартовое значение шага: `previous_alpha`, если оно передано, иначе `alpha_0`;
* `'Wolfe'` — сильные условия Вульфа: воспользуйтесь библиотечной функцией `scipy.optimize.line_search`, передав ей `c1` и `c2` из параметров объекта. Функция принимает саму функцию и её градиент (`oracle.func` и `oracle.grad`), а также точку `x_k` и направление `d_k`; найденный шаг лежит в поле `.alpha` возвращаемого результата (в старых версиях scipy возвращается кортеж, и шаг — это элемент с индексом 0).

**Важно.** У `scipy.optimize.line_search` есть один недостаток: она иногда не сходится, и в возвращаемом результате `alpha` оказывается равным `None`. Если это произошло, запустите процедуру дробления шага для поиска точки, удовлетворяющей условию Армихо (т. е. сделайте переход к стратегии `'Armijo'`).

В стратегиях Армихо (и в fallback после Вульфа) значения функции и её производной вдоль направления $\varphi_k(\alpha)$ вычисляйте через методы оракула `func_directional` и `grad_directional` — они уже реализованы в базовом классе `BaseSmoothOracle`.

**Как проверить себя:**

```
python -m pytest presubmit_tests.py::test_line_search
```

## 1.4 Градиентный спуск

Градиентный спуск
$$x_{k+1} = x_k - \alpha_k \nabla f(x_k)$$
можно рассматривать как метод спуска, в котором направление поиска $d_k$ равно антиградиенту $-\nabla f(x_k)$, а длина шага $\alpha_k$ выбирается с помощью линейного поиска.

### Задание 2. Градиентный спуск — файл `optimization.py`

**Что нужно сделать.** Реализуйте функцию `gradient_descent` — градиентный спуск по общей схеме из раздела 1.1:

* **направление:** $d_k = -\nabla f(x_k)$;
* **шаг:** $\alpha_k$ подбирается объектом `LineSearchTool` (задание 1). Используйте адаптивный подбор шага из раздела 1.3: на итерации $k \geq 1$ передавайте в линейный поиск `previous_alpha=2 * alpha_{k-1}`;
* **критерий остановки:** критерий (1.1), т. е. $\|\nabla f(x_k)\|_2^2 \leq \text{tolerance} \cdot \|\nabla f(x_0)\|_2^2$. Проверяйте его в начале каждой итерации;
* **превышение числа итераций:** если после `max_iter` итераций критерий так и не выполнен, возвращайте сообщение `'iterations_exceeded'`;
* **ошибки вычислений:** если в процессе вычислений появились значения `None`, `inf` или `nan` (например, из-за переполнения), возвращайте сообщение `'computational_error'`.

Функция возвращает кортеж `(x_star, message, history)`. Форматы возвращаемых значений и структура словаря `history` при `trace=True` подробно описаны в docstring функции — следуйте им. Не забудьте про параметр `display` для отладочного вывода.

**Как проверить себя:**

```
python -m pytest presubmit_tests.py::test_gd_basic presubmit_tests.py::test_gd_1d
```

# Часть 2. Эксперименты с градиентным спуском

В этой части проводятся эксперименты с реализованным градиентным спуском на квадратичных функциях. Оракул квадратичной функции `QuadraticOracle` уже полностью реализован в модуле `oracles`. Он соответствует функции
$$f(x) = \frac{1}{2}\langle Ax, x \rangle - \langle b, x \rangle, \qquad A \in S_{++}^n, \; b \in \mathbb{R}^n.$$

Для рисования линий уровня функции используйте функцию `plot_levels`, а для рисования траекторий метода — функцию `plot_trajectory` из файла `plot_trajectory_2d.py`.

Результаты экспериментов из этой и следующих частей оформляйте в отчёте (требования к отчёту — в последнем разделе).

### Задание 3. Эксперимент: траектория градиентного спуска на квадратичной функции

**Что нужно сделать.** Придумайте две-три квадратичные **двумерные** функции, на которых работа градиентного спуска будет заметно отличаться (например, с разным числом обусловленности). Для каждой из них запустите градиентный спуск из нескольких начальных точек и с разными стратегиями выбора шага (константная, Армихо, Вульф). Нарисуйте графики с линиями уровня функций и траекториями методов (функции `plot_levels` и `plot_trajectory`).

**Вопрос для анализа.** Как отличается поведение метода в зависимости от числа обусловленности функции, выбора начальной точки и стратегии выбора шага? Ответьте на него в отчёте.

### Задание 4. Эксперимент: число итераций градиентного спуска в зависимости от числа обусловленности и размерности

**Что нужно сделать.** Исследуйте, как зависит число итераций $T(n, \kappa)$, необходимое градиентному спуску для сходимости (успешному выходу по критерию остановки), от двух параметров: 1) числа обусловленности $\kappa \geq 1$ оптимизируемой функции и 2) размерности пространства $n$.

Для заданных $n$ и $\kappa$ сгенерируйте случайную квадратичную задачу размера $n$ с числом обусловленности $\kappa$ и запустите на ней градиентный спуск с некоторой фиксированной требуемой точностью. Замерьте число итераций, которое потребовалось методу до сходимости.

Схема эксперимента. Зафиксируйте значение размерности $n$. Переберите различные числа обусловленности $\kappa$ по сетке и постройте график зависимости $T(n, \kappa)$ от $\kappa$. Поскольку каждый раз квадратичная задача генерируется случайным образом, повторите эксперимент несколько раз — в итоге для фиксированного $n$ получится целое семейство кривых, нарисуйте их все одним цветом (например, красным). Теперь увеличьте $n$ и повторите эксперимент: новое семейство кривых нарисуйте другим цветом (например, синим). Повторите процедуру для нескольких значений $n$. Какие выводы можно сделать из полученной картинки?

**Рекомендация.** Проще всего сгенерировать случайную квадратичную задачу размера $n$ с числом обусловленности $\kappa$ следующим образом. В качестве матрицы $A \in S_{++}^n$ удобно взять просто диагональную матрицу $A = \mathrm{Diag}(a)$, у которой диагональные элементы сгенерированы случайно в пределах $[1, \kappa]$, причём $\min(a) = 1$, $\max(a) = \kappa$. В качестве вектора $b \in \mathbb{R}^n$ можно взять вектор со случайными элементами. Диагональные матрицы удобно рассматривать, поскольку с ними можно эффективно работать даже при больших значениях $n$; рекомендуется хранить матрицу $A$ в формате разреженной диагональной матрицы (см. `scipy.sparse.diags`).

Обратите внимание, что значения размерности $n$ имеет смысл перебирать по логарифмической сетке (например, $n = 10$, $n = 100$, $n = 1000$ и т. д.). Если вы используете случайность — зафиксируйте `seed`, чтобы результат был воспроизводимым.

# Часть 3. Метод Ньютона

Метод Ньютона:
$$x_{k+1} = x_k - \alpha_k [\nabla^2 f(x_k)]^{-1} \nabla f(x_k).$$

Для метода Ньютона очень важно использовать единичный шаг $\alpha_k = 1$, чтобы обеспечить локальную квадратичную сходимость. Поэтому в алгоритмах линейного поиска нужно всегда первым делом пробовать единичный шаг. Теория гарантирует, что в зоне квадратичной сходимости метода Ньютона единичный шаг будет удовлетворять условиям Армихо/Вульфа и поэтому будет приниматься автоматически. Если единичный шаг не удовлетворяет условиям Армихо/Вульфа, то алгоритмы линейного поиска его уменьшат и тем самым обеспечат глобальную сходимость метода Ньютона.

Вычисление ньютоновского направления $d_k = -[\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$ эквивалентно решению линейной системы уравнений
$$\nabla^2 f(x_k)\, d_k = -\nabla f(x_k).$$
Если гессиан — положительно определённая матрица ($\nabla^2 f(x_k) \succ 0$), то предпочтительным методом решения такой системы является разложение Холецкого: как и метод Гаусса, оно работает за $O(n^3)$, но является вычислительно более эффективным. Если матрица системы не является положительно определённой, то разложение Холецкого сможет обнаружить это и сообщить (бросить исключение).

### Задание 5. Метод Ньютона — файл `optimization.py`

**Что нужно сделать.** Реализуйте функцию `newton` по аналогии с `gradient_descent` из задания 2. Отличия:

* **направление:** $d_k = -[\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$. Для его вычисления решайте линейную систему $\nabla^2 f(x_k)\, d_k = -\nabla f(x_k)$;
* **линейный поиск:** на каждой итерации начинайте подбор шага с единичного, $\alpha_k^{(0)} = 1$ (см. теорию выше);
* **новое сообщение об ошибке:** если гессиан в текущей точке не оказался положительно определённым и систему не удалось решить, возвращайте `'newton_direction_error'`.

Остальное — критерий остановки (1.1), заполнение `history` при `trace=True`, сообщения `'iterations_exceeded'` и `'computational_error'` — так же, как в градиентном спуске; точные форматы описаны в docstring функции.

**Важно.** Для поиска направления не нужно обращать гессиан в явном виде (функция `np.linalg.inv`) и не нужно использовать самый общий метод решения системы линейных уравнений (`numpy.linalg.solve`). Учтите тот факт, что в рассматриваемой задаче гессиан является симметричной положительно определённой матрицей, и воспользуйтесь разложением Холецкого (функции `scipy.linalg.cho_factor` и `scipy.linalg.cho_solve`).

**Как проверить себя:**

```
python -m pytest presubmit_tests.py::test_newton_basic presubmit_tests.py::test_newton_1d presubmit_tests.py::test_newton_fail
```

# Часть 4. Логистическая регрессия

## 4.1 Двухклассовая логистическая регрессия

Логистическая регрессия является стандартной моделью в задачах классификации. Для простоты рассмотрим лишь случай бинарной классификации. Неформально задача формулируется следующим образом. Имеется обучающая выборка $(a_i, b_i)_{i=1}^m$, состоящая из $m$ векторов $a_i \in \mathbb{R}^n$ (называемых признаками) и соответствующих им чисел $b_i \in \{-1, 1\}$ (называемых классами). Нужно построить алгоритм $b(\cdot)$, который для произвольного нового вектора признаков $a$ автоматически определит его класс $b(a) \in \{-1, 1\}$.

В модели логистической регрессии определение класса выполняется по знаку линейной комбинации компонент вектора $a$ с некоторыми фиксированными коэффициентами $x \in \mathbb{R}^n$:
$$b(a) := \mathrm{sign}(\langle a, x \rangle).$$

Коэффициенты $x$ являются параметрами модели и настраиваются с помощью решения следующей оптимизационной задачи:
$$\underset{x \in \mathbb{R}^n}{\min} \left( \frac{1}{m} \sum_{i=1}^m \ln(1 + \exp(-b_i \langle a_i, x \rangle)) + \frac{\lambda}{2} \|x\|_2^2 \right),$$
где $\lambda > 0$ — коэффициент регуляризации (параметр модели).

### Задание 6. Формулы для функции, градиента и гессиана — в отчёт

**Что нужно сделать.** Получите формулы для градиента и гессиана функции логистической регрессии из раздела 4.1. Выпишите их в отчёт **в матрично-векторной форме**, с использованием поэлементных функций, но без каких-либо суммирований. Также выпишите в отчёт выражение для самой функции логистической регрессии в матрично-векторной форме (без явных суммирований).

**Замечание.** В матрично-векторной форме допускается использование операций матричного сложения/произведения, умножения на скаляр, транспонирования, стандартного скалярного произведения, поэлементного произведения, а также применения ко всем элементам вектора некоторой скалярной функции. Кроме этого, допускается использование стандартных матриц/векторов (заданного размера): единичная матрица $I_n$, нулевая матрица $0_{m \times n}$, нулевой вектор $0_n$, вектор из всех единиц $1_n := (1, \ldots, 1)$.

Свои выкладки можно сверить со справкой, приведённой ниже.


## Справка: функция, градиент и гессиан логистической регрессии

Ниже приведены формулы из задания 6 — пользуйтесь ими как шпаргалкой при реализации оракула (задание 7).

Обозначим $z := b \odot (Ax) \in \mathbb{R}^m$, где $\odot$ — поэлементное умножение, а все скалярные функции ($\ln$, $\exp$, $\sigma$) применяются к векторам поэлементно. Тогда

$$f(x) = \frac{1}{m}\, \mathbf{1}_m^T \ln\!\big(1 + e^{-z}\big) + \frac{\lambda}{2} \|x\|_2^2,$$

$$\nabla f(x) = -\frac{1}{m}\, A^T \big( b \odot \sigma(-z) \big) + \lambda x, \qquad \sigma(t) := \frac{1}{1 + e^{-t}},$$

$$\nabla^2 f(x) = \frac{1}{m}\, A^T\, \operatorname{Diag}(s)\, A + \lambda I_n, \qquad s := \sigma(-z) \odot \big(1 - \sigma(-z)\big).$$

**Как считать это в numpy/scipy (без переполнений и с поддержкой разреженных матриц):**

* $\ln(1 + e^{-z})$ — функция `np.logaddexp(0, -z)`: она эквивалентна «в лоб», но не переполняется при больших $|z|$;
* сигмоида $\sigma(z)$ — функция `scipy.special.expit(z)`;
* произведения $Ax$ и $A^T y$ — методы `A.dot(x)` и `A.T.dot(y)`: они одинаково работают и для плотных матриц `np.array`, и для разреженных `scipy.sparse.csr_matrix`;
* в гессиане произведение $A^T\, \operatorname{Diag}(s)\, A$ не нужно считать через явную диагональную матрицу — достаточно поэлементно домножить строки $A$ на вектор $s$: для плотной матрицы это `A * s[:, None]`, для разреженной — метод `A.multiply(s[:, None])` (тип матрицы проверяется функцией `scipy.sparse.issparse`);
* слагаемое $\lambda I_n$ — это `regcoef * np.eye(n)`.

## 4.2 Разностная проверка градиента и гессиана

Проверить правильность реализации подсчёта градиента можно с помощью конечных разностей:
$$[\nabla f(x)]_i \approx \frac{f(x + \varepsilon_1 e_i) - f(x)}{\varepsilon_1},$$
где $e_i := (0, \ldots, 0, 1, 0, \ldots, 0)$ — $i$-й базисный орт, а $\varepsilon_1$ — достаточно маленькое положительное число: $\varepsilon_1 \sim \sqrt{\varepsilon_{mach}}$, где $\varepsilon_{mach}$ — машинная точность ($\approx 10^{-16}$ для типа `double`).

Вторые производные:
$$[\nabla^2 f(x)]_{ij} \approx \frac{f(x + \varepsilon_2 e_i + \varepsilon_2 e_j) - f(x + \varepsilon_2 e_i) - f(x + \varepsilon_2 e_j) + f(x)}{\varepsilon_2^2},$$
здесь $\varepsilon_2 \sim \sqrt[3]{\varepsilon_{mach}}$.

### Задание 7. Оракул логистической регрессии и разностные производные — файл `oracles.py`

**Что нужно сделать.**

1. Реализуйте оракул логистической регрессии: методы `func`, `grad` и `hess` класса `LogRegL2Oracle` (используйте формулы из задания 6). Оракул создаётся напрямую по данным: `LogRegL2Oracle(A, b, regcoef)`.
2. Реализуйте функции `grad_finite_diff` и `hess_finite_diff` — разностные аппроксимации градиента и гессиана (формулы из раздела 4.2).
3. Проверьте правильность реализации оракула: сгенерируйте небольшую модельную выборку (матрицу $A$ и вектор $b$) и сравните значения, выдаваемые методами `grad` и `hess`, с соответствующими разностными аппроксимациями в нескольких пробных точках $x$. Эту проверку удобно оформить отдельной ячейкой и привести в отчёте.

**Замечания.**

* Реализация оракула должна быть полностью векторизованной, т. е. код не должен содержать никаких циклов по элементам выборки.
* Ваш код должен поддерживать как плотные матрицы $A$ типа `np.array`, так и разреженные типа `scipy.sparse.csr_matrix` (конкретные приёмы — в справке выше).
* Нигде в промежуточных вычислениях не стоит вычислять значение $\exp(-b_i \langle a_i, x \rangle)$, иначе может произойти переполнение. Вместо этого следует напрямую вычислять необходимые величины с помощью специализированных функций: `np.logaddexp` для $\ln(1 + \exp(\cdot))$ и `scipy.special.expit` для $1 / (1 + \exp(\cdot))$.

**Как проверить себя:**

```
python -m pytest presubmit_tests.py::test_log_reg_usual presubmit_tests.py::test_grad_finite_diff_1 presubmit_tests.py::test_grad_finite_diff_2 presubmit_tests.py::test_hess_finite_diff_1 presubmit_tests.py::test_hess_finite_diff_2
```

# Часть 5. Финальный эксперимент: сравнение методов на реальных данных

Все методы и оракул логистической регрессии реализованы — осталось сравнить градиентный спуск и метод Ньютона на реальной задаче обучения логистической регрессии.

### Задание 8. Эксперимент: сравнение градиентного спуска и метода Ньютона на задаче логистической регрессии

**Что нужно сделать.** Сравните методы градиентного спуска и Ньютона на задаче обучения логистической регрессии на реальных данных.

В качестве реальных данных используйте следующие три набора с сайта LIBSVM: *w8a*, *gisette* и *real-sim* — они приложены к заданию в папке `data/` (страница с данными: [LIBSVM datasets](http://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/)). Коэффициент регуляризации взять стандартным образом: $\lambda = 1/m$. Параметры обоих методов взять равными параметрам по умолчанию. Начальную точку выбрать $x_0 = 0$.

Для каждого набора данных постройте графики сходимости следующих двух видов:

1. Зависимость значения функции от реального времени работы метода.
2. Зависимость относительного квадрата нормы градиента $\|\nabla f(x_k)\|_2^2 / \|\nabla f(x_0)\|_2^2$ (в логарифмической шкале) от реального времени работы.

При этом оба метода (градиентный спуск и Ньютон) нужно рисовать на одном и том же графике.

Дополнительно укажите в отчёте, какова стоимость одной итерации и сколько памяти требуется каждому из методов в зависимости от параметров $m$ (размер выборки) и $n$ (размерность пространства). При оценке используйте нотацию $O(\cdot)$, скрывающую внутри себя абсолютные константы.

**Вопросы для анализа.** Какие выводы можно сделать по результатам этого эксперимента? Какой из методов лучше и в каких ситуациях?

**Рекомендация.** Любой набор данных с сайта LIBSVM представляет из себя текстовый файл в формате svmlight. Чтобы считать такой текстовый файл, можно использовать функцию `load_svmlight_file` из модуля `sklearn.datasets`. Обратите внимание, что эта функция возвращает матрицу в формате `scipy.sparse.csr_matrix`, поэтому ваша реализация логистического оракула должна поддерживать такие матрицы.

# Оформление и проверка задания

## Что сдавать

Результатом выполнения задания являются:

1. Файлы `optimization.py` и `oracles.py` с реализованными методами и оракулами.
2. Полные исходные коды для проведения экспериментов и рисования всех графиков. Все результаты должны быть воспроизводимыми; если вы используете случайность — зафиксируйте `seed`.
3. Отчёт в формате `.ipynb` о проведённых исследованиях.

## Требования к отчёту

Каждый эксперимент (задания 3, 4 и 8) оформите в виде отдельного раздела отчёта. Для каждого эксперимента необходимо сначала написать его описание: какие функции оптимизируются, каким образом генерируются данные, какие методы и с какими параметрами используются. Далее должны быть представлены результаты соответствующего эксперимента — графики, таблицы и т. д. Наконец, после результатов должны быть написаны ваши выводы: какая зависимость наблюдается и почему.

**Важно:** отчёт не должен содержать много кода. Каждый график должен быть прокомментирован: что на нём изображено и какие выводы можно сделать из этого эксперимента. У графиков обязательно должны быть подписаны оси; если на графике нарисовано несколько кривых, должна быть легенда. Сами линии следует рисовать достаточно толстыми, чтобы они были хорошо видимыми.

## Проверка и оценивание

Перед отправкой задания обязательно убедитесь, что ваша реализация проходит автоматические предварительные тесты `presubmit_tests.py`, выданные вместе с заданием. Для этого запустите следующую команду:

```
python -m pytest presubmit_tests.py
```

**Важно:** решения, которые не будут проходить тесты `presubmit_tests.py`, будут автоматически оценены в **0 баллов**. Проверяющий не будет разбираться, почему ваш код не работает, и читать ваш отчёт.

Оценка за задание будет складываться из двух частей:

1. Правильность и эффективность реализованного кода — оценивается автоматически с помощью независимых тестов (отличных от предварительных).
2. Качество отчёта — оценивается проверяющим. При этом оценка может быть субъективной и апелляции не подлежит.

За реализацию модификаций алгоритмов и хорошие дополнительные эксперименты могут быть начислены дополнительные баллы. Начисление этих баллов является субъективным и безапелляционным.

**Важно:** практическое задание выполняется самостоятельно. Если вы получили ценные советы (по реализации или проведению экспериментов) от другого студента, об этом должно быть явно написано в отчёте. В противном случае «похожие» решения считаются плагиатом, и все задействованные студенты (в том числе те, у кого списали) будут сурово наказаны.